In [1]:
# --- 1. Импорты и общие настройки ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from pmdarima import auto_arima
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet  # pip install prophet
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings

warnings.filterwarnings("ignore")
# plt.style.use("seaborn-whitegrid")



In [2]:
# --- 2. Загрузка данных ---
# df должен содержать столбцы: "Регион", "Период" (YYYY-MM), и целевой столбец с показателем.
df = pd.read_excel("Датасет по молоку v2.xlsx")
df["Период"] = pd.to_datetime(df["Период"], format="%Y-%m")
df.sample(10)



,Регион,Период,Молоко
473,АТЫРАУСКАЯ ОБЛАСТЬ,2022-09-01,7085.3
93,АКМОЛИНСКАЯ ОБЛАСТЬ,2022-10-01,29526.8
975,ЖАМБЫЛСКАЯ ОБЛАСТЬ,2015-01-01,14848.0
1182,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,2021-09-01,23958.7
1157,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,2019-08-01,25051.3
792,ГАСТАНА,2017-07-01,70.0
829,ГАСТАНА,2020-08-01,27.2
962,ГШЫМКЕНТ,2024-07-01,7373.9
1036,ЖАМБЫЛСКАЯ ОБЛАСТЬ,2020-02-01,16995.0
375,АЛМАТИНСКАЯ ОБЛАСТЬ,2025-02-01,16396.4


In [3]:
# === загружаем данные ===
best_methods = pd.read_excel("results/Молоко - Лучшие модели (MAPE_then_MAE) v2.xlsx")  # лучшие методы
best_methods

,Регион,MAPE_HW,MAE_HW,MAPE_SARIMA,MAE_SARIMA,MAPE_Prophet,MAE_Prophet,Best_method,Best_criterion,Best_MAPE,Best_MAE
0,АКМОЛИНСКАЯ ОБЛАСТЬ,9.43,1744.53,9.96,1639.63,19.57,3229.36,HW,MAPE,9.43,1639.63
1,АКТЮБИНСКАЯ ОБЛАСТЬ,9.94,1284.40,11.12,1202.32,22.16,2706.96,HW,MAPE,9.94,1202.32
2,АТЫРАУСКАЯ ОБЛАСТЬ,7.50,192.97,10.91,194.89,27.40,689.50,HW,MAPE,7.50,192.97
3,ЖАМБЫЛСКАЯ ОБЛАСТЬ,3.46,562.66,5.17,714.89,12.86,2193.30,HW,MAPE,3.46,562.66
4,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,8.46,1246.99,10.33,1811.75,9.28,1612.72,HW,MAPE,8.46,1246.99
5,КАРАГАНДИНСКАЯ ОБЛАСТЬ,4.43,826.93,4.64,754.61,22.41,3528.97,HW,MAPE,4.43,754.61
6,КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ,11.89,483.86,27.65,1376.51,17.88,739.61,HW,MAPE,11.89,483.86
7,ОБЛАСТЬ ҰЛЫТАУ,12.33,558.32,15.61,867.17,46.63,2366.68,HW,MAPE,12.33,558.32
8,ГАЛМАТЫ,63.09,17.59,81.36,22.93,54.61,14.98,Prophet,MAPE,54.61,14.98
9,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,9.55,3796.46,11.39,4539.20,8.23,3119.19,Prophet,MAPE,8.23,3119.19


In [4]:
actual_aug = pd.read_excel("Молоко 08.2025.xlsx")
actual_aug["Период"] = pd.to_datetime(actual_aug["Период"], format="%Y-%m")
actual_aug["Молоко"] = (actual_aug["Молоко"]
                     .astype(str)
                     .str.replace(".", "", regex=False)   # убираем разделители тысяч
                     .str.replace(",", ".", regex=False)  # заменяем запятую на точку
                     .astype(float))
actual_aug.to_excel("Молоко обработанные август 2025.xlsx", index=False)
actual_aug



,Регион,Период,Молоко
0,АКМОЛИНСКАЯ ОБЛАСТЬ,2025-08-01,22841.2
1,АКТЮБИНСКАЯ ОБЛАСТЬ,2025-08-01,20383.1
2,АЛМАТИНСКАЯ ОБЛАСТЬ,2025-08-01,32885.3
3,АТЫРАУСКАЯ ОБЛАСТЬ,2025-08-01,3499.3
4,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,2025-08-01,20799.6
5,ЖАМБЫЛСКАЯ ОБЛАСТЬ,2025-08-01,25297.6
6,КАРАГАНДИНСКАЯ ОБЛАСТЬ,2025-08-01,18818.5
7,КОСТАНАЙСКАЯ ОБЛАСТЬ,2025-08-01,14383.9
8,КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ,2025-08-01,4327.9
9,МАНГИСТАУСКАЯ ОБЛАСТЬ,2025-08-01,NaN


In [5]:
# === настройки ===
TARGET = "Молоко"
CUTOFF = "2025-07-01"
FORECAST = "2025-08-01"
EPS = 1e-6
SEAS = 12

In [6]:
# оставляем только август 2025 для проверки
fact_aug = (actual_aug[actual_aug["Период"] == "2025-08-01"]
            .set_index("Регион")[TARGET])


In [7]:
# === функции прогнозов, строго как в обучающем коде ===
def fc_hw_like_training(train):
    # train — Series с MS частотой
    train_log = np.log1p(train)  # log1p
    model = ExponentialSmoothing(train_log, seasonal="add", seasonal_periods=SEAS)\
            .fit(optimized=True)
    fc_log = model.forecast(1)
    return float(np.expm1(fc_log).iloc[0])  # expm1

In [8]:
def fc_sarima_like_training(train):
    train_plus = train + EPS
    train_log  = np.log(train_plus)
    use_seasonal = len(train_log) >= 2 * SEAS

    sar = auto_arima(
        train_log,
        seasonal=use_seasonal,
        m=SEAS if use_seasonal else 1,
        D=1 if use_seasonal else 0,
        seasonal_test=None,
        boxcox=True,            # как в обучении
        stepwise=True,
        suppress_warnings=True,
        error_action="ignore"
    )

    fc_log = sar.predict(n_periods=1)
    # берём первый элемент позиционно, независимо от типа (Series/ndarray/scalar)
    fc_log_scalar = np.asarray(fc_log).ravel()[0]

    return float(np.exp(fc_log_scalar) - EPS)

In [9]:
def fc_prophet_like_training(train):
    df_p = (train.reset_index()
                 .rename(columns={"Период": "ds", TARGET: "y"}))
    df_p["y"] = np.log(df_p["y"] + EPS)       # лог как в обучении
    m = Prophet()
    m.fit(df_p)
    future = m.make_future_dataframe(periods=1, freq="MS")
    yhat_log = m.predict(future)["yhat"].iloc[-1]
    return float(np.exp(yhat_log) - EPS)

In [10]:
methods_map = {
    "HW": fc_hw_like_training,
    "Holt-Winters": fc_hw_like_training,
    "Holt_Winters": fc_hw_like_training,
    "SARIMA": fc_sarima_like_training,
    "Prophet": fc_prophet_like_training,
}

In [11]:
# === прогон по регионам согласно «лучшему методу» ===
rows = []
for _, r in best_methods.iterrows():
    region = r["Регион"]
    method = r["Best_method"]

    ts = (df[df["Регион"] == region]
          .set_index("Период")[TARGET]
          .asfreq("MS")
          .sort_index())

    train = ts[:CUTOFF].dropna()
    if len(train) < 24:
        # как и в обучении, пропускаем короткие ряды
        continue

    # вызов нужной функции
    f = methods_map.get(method)
    if f is None:
        # на всякий случай нормализуем ключи
        key = str(method).strip().upper()
        if key == "HW" or "HOLT" in key:
            f = fc_hw_like_training
        elif "SARIMA" in key or "ARIMA" in key:
            f = fc_sarima_like_training
        else:
            f = fc_prophet_like_training

    fc = f(train)
    actual = fact_aug.get(region, np.nan)
    pct_dev = (fc - actual) / actual * 100 if pd.notna(actual) else np.nan

    rows.append({
        "Регион": region,
        "Лучший метод": method,
        "Прогноз (2025-08)": round(fc, 2),
        "Факт (2025-08)": round(actual, 2) if pd.notna(actual) else np.nan,
        "Отклонение, %": round(pct_dev, 2) if pd.notna(pct_dev) else np.nan
    })

results_aug = pd.DataFrame(rows).sort_values("Регион").reset_index(drop=True)
results_aug.to_excel("results/Молоко - Прогноз на 2025-08 (как в обучении).xlsx")
results_aug

18:52:36 - cmdstanpy - INFO - Chain [1] start processing
18:52:36 - cmdstanpy - INFO - Chain [1] done processing
18:52:36 - cmdstanpy - INFO - Chain [1] start processing
18:52:36 - cmdstanpy - INFO - Chain [1] done processing


,Регион,Лучший метод,Прогноз (2025-08),Факт (2025-08),"Отклонение, %"
0,АКМОЛИНСКАЯ ОБЛАСТЬ,HW,22099.24,22841.2,-3.25
1,АКТЮБИНСКАЯ ОБЛАСТЬ,HW,18628.43,20383.1,-8.61
2,АЛМАТИНСКАЯ ОБЛАСТЬ,SARIMA,33534.50,32885.3,1.97
3,АТЫРАУСКАЯ ОБЛАСТЬ,HW,3294.95,3499.3,-5.84
4,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,SARIMA,25783.49,25997.1,-0.82
5,ГАЛМАТЫ,Prophet,19.44,NaN,NaN
6,ГАСТАНА,SARIMA,20.39,NaN,NaN
7,ГШЫМКЕНТ,SARIMA,5974.24,NaN,NaN
8,ЖАМБЫЛСКАЯ ОБЛАСТЬ,HW,25641.78,25297.6,1.36
9,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,HW,22798.31,20799.6,9.61
